### 1. Introduction and Setup
In this notebook, we will develop a simple next-token prediction model using a custom-built Transformer architecture. We'll use basic self-attention, build our own tokenization, and train it on a small sample text.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Set random seed for reproducibility
torch.manual_seed(42)

### 2. Dataset and Vocabulary Preparation
We will define a simple corpus of text, tokenize it at the character level, and create mapping dictionaries.

In [7]:
# Sample corpus
corpus = """To be, or not to be, that is the question:
Whether 'this nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them."""

# Create character-level vocabulary
chars = sorted(list(set(corpus)))
vocab_size = len(chars)
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {''.join(chars)}")

Vocabulary size: 31
Vocabulary: 
 ',.:AOTWabdefghiklmnopqrstuwy


### 3. Basic Self-Attention and Transformer Layer
Here we implement a single-head scaled dot-product attention module with causal masking, followed by a Transformer Block.

In [8]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.q_projection = nn.Linear(d_model, d_model)
        self.k_projection = nn.Linear(d_model, d_model)
        self.v_projection = nn.Linear(d_model, d_model)
        self.out_projection = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.size() # Batch size, sequence length, embedding dim

        # Project inputs to query, key, and value
        q = self.q_projection(x)
        k = self.k_projection(x)
        v = self.v_projection(x)

        # Compute scaled dot-product attention
        attention_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(C)

        # Apply causal mask (lower triangular matrix) to ensure tokens only attend to previous tokens
        mask = torch.tril(torch.ones(T, T)).to(x.device)
        attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = F.softmax(attention_scores, dim=-1)

        # Weighted sum of values
        out = torch.matmul(attention_weights, v)
        return self.out_projection(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.attention = CausalSelfAttention(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Attention sublayer with residual connection and LayerNorm
        x = x + self.attention(self.ln1(x))
        # Feed-forward network sublayer with residual connection and LayerNorm
        x = x + self.ffn(self.ln2(x))
        return x

### 4. Next-Token Prediction Model
This class combines token embedding, positional embedding, our Transformer blocks, and a final linear layer to project predictions back to the vocabulary space.

In [9]:
class NextTokenTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, d_ff, max_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.transformer_block = TransformerBlock(d_model, d_ff)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.size()
        positions = torch.arange(0, T, device=idx.device).unsqueeze(0).expand(B, T)

        # Add token and positional embeddings
        x = self.token_embedding(idx) + self.position_embedding(positions)

        # Pass through transformer layers
        x = self.transformer_block(x)

        # Output logits for next-token predictions
        logits = self.fc_out(x)
        return logits

### 5. Training and Output Generation
Let's train the model using cross-entropy loss to predict the next character.

In [10]:
# Hyperparameters
MAX_LEN = len(corpus) - 1
D_MODEL = 64
D_FF = 128
EPOCHS = 150
LR = 0.01

# Prepare input (x) and target (y)
data = [char_to_ix[ch] for ch in corpus]
x_train = torch.tensor(data[:-1]).unsqueeze(0)  # Shape: (1, seq_len)
y_train = torch.tensor(data[1:]).unsqueeze(0)   # Shape: (1, seq_len)

model = NextTokenTransformer(vocab_size, D_MODEL, D_FF, MAX_LEN)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# Training Loop
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(x_train)
    loss = criterion(logits.view(-1, vocab_size), y_train.view(-1))

    loss.backward()
    optimizer.step()

    if epoch % 30 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{EPOCHS} | Loss: {loss.item():.4f}")

Epoch 1/150 | Loss: 3.7633
Epoch 30/150 | Loss: 0.0004
Epoch 60/150 | Loss: 0.0000
Epoch 90/150 | Loss: 0.0000
Epoch 120/150 | Loss: 0.0000
Epoch 150/150 | Loss: 0.0000


### 6. Sampling / Generating New Text
Let's write a function to generate a sequence character by character.

In [11]:
def generate_text(model, start_str, length):
    model.eval()
    generated = start_str
    input_seq = [char_to_ix[ch] for ch in start_str]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor(input_seq).unsqueeze(0)
            logits = model(x)
            next_token_logits = logits[0, -1, :]
            probs = F.softmax(next_token_logits, dim=-1)

            # Sample from the probability distribution
            next_token = torch.multinomial(probs, num_samples=1).item()

            generated += ix_to_char[next_token]
            input_seq.append(next_token)

            # Trim the sequence to fit positional embedding max limit if necessary
            if len(input_seq) >= MAX_LEN:
                input_seq = input_seq[-MAX_LEN:]

    return generated

# Test the text generator
print("\n--- Generated Text Response ---")
print(generate_text(model, "To be", 50))


--- Generated Text Response ---
To be, or not to be, that is the question:
Whether 'thi
